# 05 — LCEL & Runnables

We've been quietly using `|` to connect a prompt, a model, and a parser:

```python
chain = prompt | model | parser
```

That `|` is the heart of **LCEL** — the *LangChain Expression Language*. This notebook
explains the system behind it so you can compose components confidently.

The key idea: almost every LangChain building block is a **Runnable**. All Runnables
share the same interface (`invoke`, `batch`, `stream`, and their async twins). Because
they share an interface, they snap together — and the chains you build get batching,
streaming, and async **for free**.

In this notebook:

1. The `Runnable` interface.
2. `|` and `RunnableSequence`.
3. `RunnableLambda` — turn any function into a Runnable.
4. `RunnableParallel` — run branches at the same time.
5. `RunnablePassthrough` and `.assign()` — manage the data flowing through.
6. `.bind()` — pin arguments.
7. Putting it together in a realistic chain.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()
assert os.environ.get("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in your .env file first."

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.3)
print("Model ready.")

## 1. Everything is a Runnable

A **Runnable** is any object with a standard set of methods. The ones you'll use:

- `.invoke(input)` — run once.
- `.batch([inputs])` — run many concurrently.
- `.stream(input)` — yield output incrementally.
- `.ainvoke`, `.abatch`, `.astream` — async versions.

Models, prompts, parsers, retrievers — all Runnables. So is a chain you build. That
uniformity is what lets you treat a five-component pipeline exactly like a single model.

In [ ]:
# A model is a Runnable: it has invoke / batch / stream.
print("invoke:", model.invoke("Say hi in one word.").content)
print("batch :", [m.content for m in model.batch(["Say red.", "Say blue."])])

## 2. `|` builds a `RunnableSequence`

The pipe operator chains Runnables so the output of one becomes the input of the next.
`a | b | c` produces a `RunnableSequence` that is itself a Runnable. Data flows
left to right.

In [ ]:
prompt = ChatPromptTemplate.from_template("Explain {concept} in one sentence for a beginner.")

chain = prompt | model | StrOutputParser()
print(type(chain).__name__)            # RunnableSequence
print(chain.invoke({"concept": "recursion"}))

Because the whole chain is a Runnable, it streams and batches too — you didn't write
any extra code for that.

In [ ]:
# Stream the same chain:
for piece in chain.stream({"concept": "an API"}):
    print(piece, end="", flush=True)
print()

# Batch it over several inputs:
for out in chain.batch([{"concept": "a cache"}, {"concept": "a queue"}]):
    print("-", out)

## 3. `RunnableLambda` — your own functions in a chain

Any plain Python function can join a chain by wrapping it in `RunnableLambda` (or just
using `.pipe()` / placing a function in a `|` where LangChain coerces it). This is how you
insert custom logic — preprocessing, formatting, post-processing — between components.

In [ ]:
from langchain_core.runnables import RunnableLambda

def shout(text: str) -> str:
    return text.upper() + "!!!"

# A function becomes a step in the pipeline.
chain = prompt | model | StrOutputParser() | RunnableLambda(shout)
print(chain.invoke({"concept": "a variable"}))

## 4. `RunnableParallel` — branch out, run together

Sometimes you want to do several things with the *same* input at once — e.g. summarize a
text **and** extract keywords. `RunnableParallel` (often written as a plain dict in a
chain) runs each branch concurrently and returns a dict of their results.

In [ ]:
from langchain_core.runnables import RunnableParallel

summary_chain  = ChatPromptTemplate.from_template("Summarize in one sentence:\n{text}") | model | StrOutputParser()
keywords_chain = ChatPromptTemplate.from_template("List 3 keywords (comma-separated) for:\n{text}") | model | StrOutputParser()

# A dict of Runnables runs the branches in parallel.
parallel = RunnableParallel(summary=summary_chain, keywords=keywords_chain)

text = """LangChain is a framework for developing applications powered by language models. It provides standard interfaces, prompt management, and tools for retrieval and agents."""

result = parallel.invoke({"text": text})
print("summary :", result["summary"])
print("keywords:", result["keywords"])

> **Why parallel matters.** Both branches hit the network at the same time, so the
> total wait is roughly the *slower* of the two — not their sum. For independent LLM calls
> this is a real speedup.

## 5. `RunnablePassthrough` and `.assign()` — shaping the data

Chains pass a single value from one step to the next. But you'll often need to *keep* the
original input while *adding* something computed from it. Two tools make that easy.

### `RunnablePassthrough()` — forward the input unchanged

This is the simplest Runnable there is: whatever goes in comes straight back out,
untouched. On its own that looks pointless, but it's the building block for everything
below — it's how you say *"carry this value forward as-is"* inside a bigger structure.
For example, dropped into a `RunnableParallel` it keeps the original input right next to a
freshly computed value.

In [ ]:
from langchain_core.runnables import RunnablePassthrough

# By itself, a passthrough just echoes whatever it receives.
print("echo:", RunnablePassthrough().invoke({"text": text}))

# Its real job: inside a parallel, keep the input while another branch computes something.
keep_and_measure = RunnableParallel(
    original=RunnablePassthrough(),                    # the input, unchanged
    length=RunnableLambda(lambda d: len(d["text"])),   # something computed from it
)
print(keep_and_measure.invoke({"text": text}))

### `RunnablePassthrough.assign(key=runnable)` — keep everything, add a key

`.assign()` is the workhorse version. It passes the **whole** input dict through *and*
runs a Runnable to add new keys to it — nothing is lost, you only gain keys. Notice the
pattern below: we build the summarizing chain in its own variable first, then hand that
variable to `.assign()`. Keeping the chain separate makes the data-shaping step easy to read.

This shows up constantly in RAG (notebook 09), where you carry the user's question
forward while also computing retrieved context from it.

In [ ]:
# 1. Build the chain that produces the value we want to add.
summary_chain = ChatPromptTemplate.from_template("Summarize in 5 words:\n{text}") | model | StrOutputParser()

# 2. .assign() runs that chain and adds its output as a "summary" key, keeping "text".
add_summary = RunnablePassthrough.assign(summary=summary_chain)

out = add_summary.invoke({"text": text})
print("kept text  :", out["text"][:50], "...")
print("new summary:", out["summary"])

## 6. `.bind()` — pin arguments to a Runnable

Inside a chain you never call the model yourself — the chain calls `.invoke()` for you. So
how do you pass a per-call option (say, a stop sequence) to the model buried inside
`prompt | model | parser`? You **bind** it ahead of time.

`.bind(**kwargs)` returns a *new* Runnable with those arguments pre-filled on every call;
nothing else changes. It's the standard way to lock model options — stop sequences, max
tokens, temperature — without building a separate model object. (Later, `.bind_tools()`
uses the same mechanism to attach tools to a model.)

In the example below we bind a **`stop` sequence**: the model halts generation the moment
it's about to produce that text. The effect is easy to see — ask it to count to 10, and a
`stop=["5"]` binding cuts the output off right before 5.

In [ ]:
# .bind() pins the `stop` option onto the model for every future call.
stop_at_five = model.bind(stop=["5"])

question = "Count from 1 to 10, one number per line."

print("--- plain model (counts all the way) ---")
print(model.invoke(question).content)

print("\n--- bound with stop=['5'] (halts before 5) ---")
print(stop_at_five.invoke(question).content)

## 7. Putting it together

Here's a small but realistic chain that mirrors how you'll structure real apps. Given a
support message, it (a) keeps the original text, (b) classifies its topic, and (c) drafts a
reply — combining a passthrough, parallel branches, and sequencing.

In [ ]:
classify = ChatPromptTemplate.from_template(
    "Classify this message into one word (billing, technical, or general):\n{message}"
) | model | StrOutputParser()

draft_reply = ChatPromptTemplate.from_template(
    "Write a polite 2-sentence reply to this customer message:\n{message}"
) | model | StrOutputParser()

pipeline = RunnableParallel(
    message=RunnablePassthrough(),
    topic=classify,
    reply=draft_reply,
)

result = pipeline.invoke("I was charged twice for my subscription this month.")
print("topic:", result["topic"].strip())
print("reply:", result["reply"])

Notice we never wrote loops, threads, or glue code — just declared how components
connect. That declarative style is the whole appeal of LCEL: chains are easy to read,
reuse, and reason about, and they inherit streaming/batch/async automatically.

## Your turn

Five exercises that wire these Runnables into small pipelines you'd actually build — an
auto-tagger, an input cleaner, a review enricher, an autocomplete, and a two-stage content
pipeline. Try each in a blank cell before opening its solution.

**Exercise 1 — Article auto-tagger.** Use `RunnableParallel` to take one `{article}` and
produce a summary, a catchy title, and five tags **at the same time**, returning a dict.

*Sample run (illustrative):*

```text
TITLE : Build LLM Apps, One Block at a Time
SUMMARY: LangChain lets developers combine model calls, retrieval, and tools into applications.
TAGS  : LangChain, LLM, retrieval, tools, orchestration
```

<details><summary>Show solution</summary>

```python
from langchain_core.runnables import RunnableParallel

summary = ChatPromptTemplate.from_template("Summarize in one sentence:\n{article}") | model | StrOutputParser()
title   = ChatPromptTemplate.from_template("Write a catchy 6-word title for:\n{article}") | model | StrOutputParser()
tags    = ChatPromptTemplate.from_template("Give 5 comma-separated tags for:\n{article}") | model | StrOutputParser()

enrich = RunnableParallel(summary=summary, title=title, tags=tags)
article = "LangChain lets developers compose model calls, retrieval, and tools into apps."
out = enrich.invoke({"article": article})
print("TITLE :", out["title"])
print("SUMMARY:", out["summary"])
print("TAGS  :", out["tags"])
```

All three branches hit the network together, so the wait is one call, not three.

</details>

**Exercise 2 — Input cleaner.** Put a `RunnableLambda` *first* in a chain to normalize messy
user text (collapse whitespace, trim, cap length) into `{"text": ...}` before a summarizer
runs.

*Sample run (illustrative):*

```text
The user typed this text with a lot of extra whitespace.
```

<details><summary>Show solution</summary>

```python
import re
from langchain_core.runnables import RunnableLambda

def clean(raw: str) -> dict:
    return {"text": re.sub(r"\s+", " ", raw).strip()[:500]}

summarize = ChatPromptTemplate.from_template("Summarize:\n{text}") | model | StrOutputParser()
chain = RunnableLambda(clean) | summarize

messy = "   the   user   typed   this \n\n\n  with  lots of   whitespace   "
print(chain.invoke(messy))
```

A `RunnableLambda` lets your own preprocessing live inside the pipeline, not outside it.

</details>

**Exercise 3 — Review enricher.** Use `RunnablePassthrough.assign` to take `{"review": ...}`
and add a `sentiment` and a suggested `reply` while keeping the original review — one row
ready for a support dashboard.

*Sample run (illustrative):*

```text
sentiment: positive
reply    : Thank you for the kind words! We're sorry about the slow shipping and are working to speed it up.
kept     : Shipping was slow but the product is excellent.
```

<details><summary>Show solution</summary>

```python
from langchain_core.runnables import RunnablePassthrough

sentiment = ChatPromptTemplate.from_template("One word — positive, negative, or neutral:\n{review}") | model | StrOutputParser()
reply     = ChatPromptTemplate.from_template("Write a one-sentence reply to this review:\n{review}") | model | StrOutputParser()

enrich = RunnablePassthrough.assign(sentiment=sentiment, reply=reply)
row = enrich.invoke({"review": "Shipping was slow but the product is excellent."})
print("sentiment:", row["sentiment"].strip())
print("reply    :", row["reply"])
print("kept     :", row["review"])
```

`.assign` adds keys without dropping the input — exactly what a dashboard row needs.

</details>

**Exercise 4 — Single-line autocomplete.** Use `.bind(stop=["\n"])` so the model returns only
one line, then build a commit-subject autocomplete that finishes a prefix.

*Sample run (illustrative — one line only):*

```text
'the response payload is missing the user id'
```

<details><summary>Show solution</summary>

```python
one_line = model.bind(stop=["\n"])
suggest = ChatPromptTemplate.from_template(
    "Complete this commit message subject line, returning only the rest of the line:\n{prefix}"
) | one_line | StrOutputParser()

print(repr(suggest.invoke({"prefix": "fix: prevent crash when "})))
```

Binding `stop` keeps the suggestion to a single line — ideal for inline completions.

</details>

**Exercise 5 — Two-stage content pipeline.** Compose a sequence *into* a parallel: first draft
a short blog intro from `{topic}`, then branch into a title and a tweet built from that draft,
returning all three.

*Sample run (illustrative):*

```text
TITLE: Vectors: The Hidden Engine Behind Smart Search
TWEET: Ever wonder how apps find meaning, not just keywords? Vector databases turn text into
numbers so search understands intent. Here's why they matter. 🔍 #AI #search
DRAFT: Vector databases store information as numerical embeddings rather than plain text. ...
```

<details><summary>Show solution</summary>

```python
from langchain_core.runnables import RunnablePassthrough

draft = ChatPromptTemplate.from_template("Write a 4-sentence blog intro about {topic}.") | model | StrOutputParser()
title = ChatPromptTemplate.from_template("Give a punchy title for this intro:\n{draft}") | model | StrOutputParser()
tweet = ChatPromptTemplate.from_template("Turn this intro into one tweet:\n{draft}") | model | StrOutputParser()

# Stage 1 produces {"draft": ...}; stage 2 adds title + tweet computed from it.
pipeline = {"draft": draft} | RunnablePassthrough.assign(title=title, tweet=tweet)

out = pipeline.invoke({"topic": "vector databases"})
print("TITLE:", out["title"])
print("TWEET:", out["tweet"])
print("DRAFT:", out["draft"][:80], "...")
```

The output of the first stage flows into the second — a sequence feeding a parallel.

</details>

## Summary

- A **Runnable** is anything with `invoke`/`batch`/`stream` (+ async). Models, prompts,
  parsers, and chains all qualify.
- **`|`** composes Runnables into a `RunnableSequence`; output flows left to right.
- **`RunnableLambda`** wraps any function so your own code fits in a chain.
- **`RunnableParallel`** (a dict of Runnables) runs branches concurrently.
- **`RunnablePassthrough`** forwards input; **`.assign()`** adds keys while keeping the rest.
- **`.bind()`** fixes arguments on a Runnable.
- Chains inherit streaming, batching, and async automatically.

**Next:** [06 — Memory & Conversation](06_Memory_and_Conversation.ipynb). We'll use these
composition tools to give a chatbot memory across turns.